# Notebook 3: GA-based Black-Box Adversarial Attack

**Goal:** Use Genetic Algorithm to generate adversarial examples WITHOUT accessing model weights.

**Key difference from FGSM/PGD:** We only query the model input→output, no gradients needed.

**Output:** GA attack success rates, comparison with baselines, fitness evolution plots.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys
sys.path.append('../')

from src.model import CNN
from src.ga import GeneticAttack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CNN().to(device)
model.load_state_dict(torch.load('../results/model.pth', map_location=device))
model.eval()
print(f'Model loaded. Device: {device}')

In [ ]:
# Load test data
transform = transforms.Compose([
    transforms.ToTensor(),
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=False, transform=transform)
classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [ ]:
# Initialize GA attacker
ga = GeneticAttack(
    model=model,
    population_size=50,
    generations=30,
    mutation_rate=0.1,
    mutation_strength=0.05,
    epsilon=0.1,
    device=device
)

print('GA attacker initialized.')
print(f'Population: {ga.population_size} | Generations: {ga.generations} | Epsilon: {ga.epsilon}')

In [ ]:
# Attack a single image and visualize fitness evolution
image, true_label = testset[0]
image_np = image.numpy()  # (C, H, W)

print(f'Attacking image of class: {classes[true_label]}')

perturbation, history = ga.attack(image_np, true_label)

# Plot fitness evolution
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('GA Fitness Evolution')
plt.grid(True)
plt.savefig('../results/figures/ga_fitness_evolution.png', dpi=150)
plt.show()

In [ ]:
# Visualize: original vs adversarial
adv_image = np.clip(image_np + perturbation, 0, 1)

adv_tensor = torch.FloatTensor(adv_image).unsqueeze(0).to(device)
orig_tensor = image.unsqueeze(0).to(device)

with torch.no_grad():
    orig_pred = model(orig_tensor).argmax(dim=1).item()
    adv_pred  = model(adv_tensor).argmax(dim=1).item()

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(image_np.transpose(1,2,0))
axes[0].set_title(f'Original: {classes[true_label]}')
axes[1].imshow(perturbation.transpose(1,2,0) * 5 + 0.5)  # amplified for visibility
axes[1].set_title('Perturbation (amplified)')
axes[2].imshow(adv_image.transpose(1,2,0))
axes[2].set_title(f'Adversarial: {classes[adv_pred]}')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('../results/figures/ga_adversarial_example.png', dpi=150)
plt.show()

print(f'Original prediction: {classes[orig_pred]}')
print(f'Adversarial prediction: {classes[adv_pred]}')
print(f'Attack success: {orig_pred != adv_pred}')

In [ ]:
# Evaluate GA attack success rate on N samples
N_SAMPLES = 100  # increase to 500 for final results
success_count = 0
perturbation_norms = []

for i in tqdm(range(N_SAMPLES)):
    image, true_label = testset[i]
    image_np = image.numpy()

    # Only attack correctly classified images
    with torch.no_grad():
        pred = model(image.unsqueeze(0).to(device)).argmax(dim=1).item()
    if pred != true_label:
        continue

    perturbation, _ = ga.attack(image_np, true_label)
    adv = np.clip(image_np + perturbation, 0, 1)

    with torch.no_grad():
        adv_pred = model(torch.FloatTensor(adv).unsqueeze(0).to(device)).argmax(dim=1).item()

    if adv_pred != true_label:
        success_count += 1

    perturbation_norms.append(np.linalg.norm(perturbation))

asr = success_count / N_SAMPLES
avg_norm = np.mean(perturbation_norms)
print(f'GA Attack Success Rate: {asr:.3f}')
print(f'Average perturbation norm: {avg_norm:.4f}')